## tl;dr
최근 3개월 Search Console 페이지 보고서에서 1차 개선 URL을 제외하고, 평균순위 4~15위·노출 10회 이상인 페이지를 CTR 부족분과 노출 규모로 정렬한다.

## Context & Methods
### Key Assumptions
목표 CTR은 후보 정렬용 내부 기준이며 순위 4~5.99는 8%, 6~7.99는 6%, 8~9.99는 4%, 10~15는 2.5%를 사용한다.

In [1]:
from pathlib import Path
import csv, io, zipfile

source_zip = Path.home() / 'Downloads' / 'https___emfls.github.io_-Performance-on-Search-2026-08-01.zip'
excluded_suffixes = {
    'cheongju.html', 'romania.html', 'slovakia.html', 'qatar.html',
    'norway.html', 'serbia.html', 'slovenia.html', 'damyang.html',
    'gimpo.html', 'asan.html'
}
assert source_zip.exists(), source_zip

## Data
ZIP 안에서 `인기 페이지` 헤더를 가진 CSV를 찾아 읽는다.

In [2]:
with zipfile.ZipFile(source_zip) as archive:
    page_text = None
    for member in archive.infolist():
        text = archive.read(member).decode('utf-8-sig')
        if text.startswith('인기 페이지,'):
            page_text = text
            break
assert page_text is not None, '페이지 CSV를 찾지 못했습니다.'
source_rows = list(csv.DictReader(io.StringIO(page_text)))
len(source_rows)

66

## Results
점수는 `노출 × (1 + 목표 CTR 대비 부족 비율)`이다.

In [3]:
def target_ctr(position):
    if position < 6: return 8.0
    if position < 8: return 6.0
    if position < 10: return 4.0
    return 2.5

candidates = []
for row in source_rows:
    url = row['인기 페이지']
    clicks = int(row['클릭수'])
    impressions = int(row['노출'])
    ctr = float(row['CTR'].rstrip('%'))
    position = float(row['게재 순위'])
    if not (4 <= position <= 15 and impressions >= 10): continue
    if any(url.endswith(suffix) for suffix in excluded_suffixes): continue
    target = target_ctr(position)
    gap = max(target - ctr, 0)
    score = impressions * (1 + gap / target)
    candidates.append({**row, 'score': round(score, 2)})
def cluster_priority(url):
    if '/visa/' in url: return 0
    if '/camp/' in url: return 1
    if '/game/' in url: return 2
    return 3

candidates.sort(key=lambda x: (-x['score'], -int(x['노출']), -int(x['클릭수']), cluster_priority(x['인기 페이지']), x['인기 페이지']))
top20 = candidates[:20]
[(row['인기 페이지'], row['클릭수'], row['노출'], row['CTR'], row['게재 순위'], row['score']) for row in top20]

[('https://emfls.github.io/kor/report/visa/northmacedonia.html',
  '4',
  '104',
  '3.85%',
  '5.65',
  157.95),
 ('https://emfls.github.io/kor/report/visa/switzerland.html',
  '1',
  '97',
  '1.03%',
  '10.71',
  154.04),
 ('https://emfls.github.io/kor/report/camp/gapyeong.html',
  '4',
  '109',
  '3.67%',
  '9.94',
  117.99),
 ('https://emfls.github.io/kor/report/camp/andong.html',
  '5',
  '114',
  '4.39%',
  '9.18',
  114.0),
 ('https://emfls.github.io/kor/report/visa/nigeria.html',
  '1',
  '67',
  '1.49%',
  '9.01',
  109.04),
 ('https://emfls.github.io/kor/report/camp/busan.html',
  '4',
  '101',
  '3.96%',
  '12.95',
  101.0),
 ('https://emfls.github.io/kor/report/camp/gwangju-g.html',
  '7',
  '93',
  '7.53%',
  '7.26',
  93.0),
 ('https://emfls.github.io/kor/report/visa/rwanda.html',
  '1',
  '55',
  '1.82%',
  '9.62',
  84.97),
 ('https://emfls.github.io/kor/report/camp/goheung.html',
  '6',
  '84',
  '7.14%',
  '7.8',
  84.0),
 ('https://emfls.github.io/kor/report/visa/ukra

In [4]:
assert len(top20) == 20
assert all(4 <= float(row['게재 순위']) <= 15 for row in top20)
assert all(int(row['노출']) >= 10 for row in top20)
assert not any(any(row['인기 페이지'].endswith(s) for s in excluded_suffixes) for row in top20)
sum(int(row['노출']) for row in top20), sum(int(row['클릭수']) for row in top20)

(1208, 51)

## Takeaways
상위 10개를 2차 첫 묶음으로 실행한다. 페이지별 검색어 교차표가 없으므로 실제 수정 전에는 검색어와 현재 콘텐츠를 개별 확인한다.